# Create a Balanced Version of the CelebA Dataset Containing the Same Number of Male and Female Images

Antonio Esteves @ UMinho, Jul 2024

In [ ]:
%load_ext watermark
%watermark -a 'Antonio Esteves' -v -p torch

### Imports

In [ ]:
import pandas as pd
import numpy  as np
import random
import os
import shutil
from   pathlib           import Path
import matplotlib.pyplot as     plt
from   PIL               import Image

%matplotlib inline

### Define Necessary Folder and File Paths

In [ ]:
# Setup path to celebA dataset folder

data_path = Path("/home/esteves/datasets/celeba")

# Setup path to the folder were we will merge all images
all_data_path = data_path / "all"

# Setup path for the folder were we will store the images of the balanced dataset
balanced_data_path =  data_path / "balanced"

# Attribute file path
attr_file = data_path / "list_attr_celeba.txt"

### Create the folders 'all' and "balanced"

In [ ]:
os.makedirs(all_data_path, exist_ok=True)
os.makedirs(balanced_data_path, exist_ok=True)

### Copy all images from 'train, 'val' and 'test' to "all"

In [ ]:
list_dirs = ['test', 'val', 'train']

for d in list_dirs:

    src_path = data_path / d
    
    files = os.listdir(src_path)
    print(f'\n[INFO] Copying {len(files)} files from source folder {d} ...')
    
    for id, f in enumerate(files):

        src_file = src_path / f
        shutil.copy(src_file, all_data_path)
        if (id+1) % 100 == 0:
            print('.', end="")

## Dataset

First, let's load the attribute list using Pandas. Here, we will be focussing on one attribute only, gender, which is stored in the "Male" column, where female face images are labeled as -1, and male images are labeled as 1. Further, we are binarizing these labels to 0's and 1's. The JPEG file names associated with each image are stored in the index of the following data frame:

In [ ]:
attr_df = pd.read_csv(
    attr_file,
    sep="\s+",
    skiprows=1
)

attr_df.head()

In [ ]:
attr_names = list(attr_df.columns)

print(attr_names)

In [ ]:
# Replace "-1" by "0" in all columns

for name in attr_names:
    # create a dictionary of replacements
    replacements = {-1: 0}

    # replace values using the .map() method
    attr_df[name] = attr_df[name].map(replacements).fillna(attr_df[name])

In [ ]:
attr_df.head()

In [ ]:
for file, *row in attr_df.itertuples():
    print(file, row[:])
    if file == "000010.jpg":
        break

In [ ]:
attr_df.index.values

In [ ]:
attr_array = attr_df.to_numpy()

num_images = attr_array.shape[0]
print(f'number of images in array: {num_images}')
print(attr_array[4])

### Print frequency of selected attributes

In [ ]:
print(f"Attractive: {attr_df[attr_df['Attractive']==1].shape[0]} \
    {(100*attr_df[attr_df['Attractive']==1].shape[0])/num_images:.1f}%")
print(f"Black_Hair: {attr_df[attr_df['Black_Hair']==1].shape[0]} \
    {(100*attr_df[attr_df['Black_Hair']==1].shape[0])/num_images:.1f}%")
print(f"Blond_Hair: {attr_df[attr_df['Blond_Hair']==1].shape[0]} \
    {(100*attr_df[attr_df['Blond_Hair']==1].shape[0])/num_images:.1f}%")
print(f"Brown_Hair: {attr_df[attr_df['Brown_Hair']==1].shape[0]} \
    {(100*attr_df[attr_df['Brown_Hair']==1].shape[0])/num_images:.1f}%")
print(f"Eyeglasses: {attr_df[attr_df['Eyeglasses']==1].shape[0]} \
    {(100*attr_df[attr_df['Eyeglasses']==1].shape[0])/num_images:.1f}%")
print(f"Goatee: {attr_df[attr_df['Goatee']==1].shape[0]} \
    {(100*attr_df[attr_df['Goatee']==1].shape[0])/num_images:.1f}%")
print(f"Gray_Hair: {attr_df[attr_df['Gray_Hair']==1].shape[0]} \
    {(100*attr_df[attr_df['Gray_Hair']==1].shape[0])/num_images:.1f}%")
print(f"Male: {attr_df[attr_df['Male']==1].shape[0]} \
    {(100*attr_df[attr_df['Male']==1].shape[0])/num_images:.1f}%")
print(f"Mustache: {attr_df[attr_df['Mustache']==1].shape[0]} \
    {(100*attr_df[attr_df['Mustache']==1].shape[0])/num_images:.1f}%")
print(f"No_Beard: {attr_df[attr_df['No_Beard']==1].shape[0]} \
    {(100*attr_df[attr_df['No_Beard']==1].shape[0])/num_images:.1f}%")
print(f"Pale_Skin: {attr_df[attr_df['Pale_Skin']==1].shape[0]} \
    {(100*attr_df[attr_df['Pale_Skin']==1].shape[0])/num_images:.1f}%")
print(f"Rosy_Cheeks: {attr_df[attr_df['Rosy_Cheeks']==1].shape[0]} \
    {(100*attr_df[attr_df['Rosy_Cheeks']==1].shape[0])/num_images:.1f}%")
print(f"Smiling: {attr_df[attr_df['Smiling']==1].shape[0]} \
    {(100*attr_df[attr_df['Smiling']==1].shape[0])/num_images:.1f}%")
print(f"Young: {attr_df[attr_df['Young']==1].shape[0]} \
    {(100*attr_df[attr_df['Young']==1].shape[0])/num_images:.1f}%")
print(f"Brown_Hair: {attr_df[attr_df['Brown_Hair']==1].shape[0]} \
    {(100*attr_df[attr_df['Brown_Hair']==1].shape[0])/num_images:.1f}%")


### Balance the dataset by 'gender' using the number of images of the least represented class as the number of images of both classes (male/female)

In [ ]:
count_male = 0

for index, row in attr_df.iterrows():
    if row['Male']==1:
        count_male += 1
count_famale = num_images-count_male

In [ ]:
if (count_famale > count_male):
    min_images = count_male
else:
    min_images = count_female

print(f'Number of images in the dataset:        {num_images}')
print(f'Number of male images in the dataset:   {count_male}')
print(f'Number of female images in the dataset: {count_famale}')
print(f'Minimum number of images of male/female in the dataset: {min_images}')

In [ ]:
male_list   = []
female_list = []
for index, row in attr_df.iterrows():
    if row['Male']==1:
        male_list.append(index)
    else:
        female_list.append(index)

print(f'Size of the list with male images:   {len(male_list)}')
print(f'Size of the list with female images: {len(female_list)}')

In [ ]:
male_list   = random.sample(male_list, min_images)
female_list = random.sample(female_list, min_images)

print(f'Size of the list with male images after random smpling:   {len(male_list)}')
print(f'Size of the list with female images after random smpling: {len(female_list)}')

### Copy equal number of male and female fles from 'all' to "balanced"

In [ ]:
for f in range(len(male_list)):
    src_file = all_data_path / male_list[f]
    shutil.copy(src_file, balanced_data_path)
    src_file = all_data_path / female_list[f]
    shutil.copy(src_file, balanced_data_path)

### Remove the auxilliary folder 'all'

In [ ]:
################## CAUTION #########################
# shutil.rmtree(all_data_path)
################## CAUTION #########################